# 03 — Offline sandbox smoke (mock)

After notebooks **01** and **02**, confirm the prepared environment can drive sandbox evals **without a live LLM**:

- Load `data/runtime/prepared/prep_manifest.json`
- Run `sandbox`-equivalent mock pilots (sorter + pipeline dry machinery)

For a real local model later: `sandbox pull-models` then `sandbox pilot --local` on the host (or against compose Ollama).

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

ROOT = Path(os.environ.get("SANDBOX_ROOT") or Path.cwd())
if (ROOT / "src").is_dir():
    sys.path.insert(0, str(ROOT / "src"))
    os.environ.setdefault("SANDBOX_ROOT", str(ROOT))

from mailroom_sandbox.prep import ensure_dotenv_from_example, prepare_offline_datasets, prepared_dir
from mailroom_sandbox.runtime import activate

ensure_dotenv_from_example()
activate(os.environ.get("SANDBOX_PROFILE", "ollama"))
os.environ["SANDBOX_RUN_MODE"] = "mock"
os.environ.setdefault("OBSERVABILITY_PROVIDER", "none")

manifest_path = prepared_dir() / "prep_manifest.json"
if not manifest_path.is_file():
    print("prepared corpus missing — running prepare_offline_datasets()…")
    prepare_offline_datasets()

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({"fingerprint": manifest["fingerprint"], "counts": manifest["counts"]}, indent=2))

## Mock sorter eval (fixture catalog)

In [ ]:
print(json.dumps(sorter, indent=2, default=str))
assert sorter["n"] >= 1
assert sorter["scores"]["exact_match"] == 1.0


## Mock pipeline pilot

In [ ]:
pipeline = run_pipeline_eval(
    mock=True,
    sample=min(4, int(manifest["counts"]["fixtures"])),
    dry_run=False,
    experiment_name="notebook_offline_pipeline",
    profile=os.environ.get("SANDBOX_PROFILE", "ollama"),
    connected=True,
)
print(json.dumps(pipeline, indent=2, default=str))

## Prepared balanced sorter slice

Sanity-check the class-balanced JSONL written by notebook 02.

In [ ]:
from mailroom_sandbox.datasets import load_jsonl
from mailroom_sandbox.paths import repo_root

balanced_rel = manifest["artifacts"]["sorter_pilot_balanced"]
balanced = load_jsonl(repo_root() / balanced_rel)
classes = sorted({r["expected_doc_class"] for r in balanced})
print(f"{len(balanced)} rows across {len(classes)} classes:", classes)
assert len(classes) == len(balanced), "balanced slice should be one row per class"

## Done

Offline path is ready. Optional next steps on the host:

```bash
sandbox up
sandbox pull-models
sandbox health
sandbox pilot --local
```